In [ ]:
%%sql -r dataframe_1
-- =====================================================
-- PHASE 3A: MEMBER STAGING TABLE
-- NOTEBOOK: NB_03_MEMBER_PROCESSING
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;
USE SCHEMA STG;

CREATE TABLE IF NOT EXISTS STG_MEMBER_PROFILES (
    -- Member attributes
    MEMBER_NAME             VARCHAR(255),
    MEMBER_ID               VARCHAR(18),
    ENROLLMENT_DATE         DATE,
    LAST_FLIGHT_DATE        DATE,
    TIER_CODE               CHAR(5),
    AGENT_NAME              VARCHAR(255),
    STATE                   CHAR(5),
    COUNTRY                 CHAR(5),
    POST_CODE               NUMBER(5,0),
    DOB                     DATE,
    ACTIVE_MEMBER           CHAR(1),

    -- Derived attributes
    AGE                     NUMBER(3,0),
    STALE_MEMBER            BOOLEAN,

    -- Source metadata
    SOURCE_FILE_NAME        VARCHAR(500),
    FILE_TIMESTAMP          TIMESTAMP_NTZ,
    INGESTION_TIMESTAMP     TIMESTAMP_NTZ,
    BATCH_ID                VARCHAR(100),
    SOURCE_ROW_NUMBER       NUMBER(38,0),

    -- Data quality fields
    VALIDATION_STATUS       VARCHAR(20),
    REJECTION_REASON        VARCHAR(2000),
    PROCESSED_TIMESTAMP     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Verify staging table
DESCRIBE TABLE STG_MEMBER_PROFILES;

SHOW TABLES IN SCHEMA STG;

In [ ]:
%%sql -r dataframe_2
-- =====================================================
-- PHASE 3B: CORE MEMBER TABLES
-- NOTEBOOK: NB_03_MEMBER_PROCESSING
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;
USE SCHEMA CORE;

-- 1. Central member master table
CREATE TABLE IF NOT EXISTS CORE.MEMBER_PROFILE (
    MEMBER_NAME                 VARCHAR(255) NOT NULL,
    MEMBER_ID                   VARCHAR(18) NOT NULL,
    ENROLLMENT_DATE             DATE NOT NULL,
    LAST_FLIGHT_DATE            DATE,
    TIER_CODE                   CHAR(5),
    AGENT_NAME                  VARCHAR(255),
    STATE                       CHAR(5),
    COUNTRY                     CHAR(5),
    POST_CODE                   NUMBER(5,0),
    DOB                         DATE,
    ACTIVE_MEMBER               CHAR(1),

    AGE                         NUMBER(3,0),
    STALE_MEMBER                BOOLEAN,

    SOURCE_FILE_NAME            VARCHAR(500),
    FILE_TIMESTAMP              TIMESTAMP_NTZ,
    INGESTION_TIMESTAMP         TIMESTAMP_NTZ,
    BATCH_ID                    VARCHAR(100),
    SOURCE_ROW_NUMBER           NUMBER(38,0),

    RECORD_CREATED_TIMESTAMP    TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    RECORD_UPDATED_TIMESTAMP    TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- 2. Country-specific member tables
CREATE TABLE IF NOT EXISTS CORE.MEMBER_PROFILE_IND
    LIKE CORE.MEMBER_PROFILE;

CREATE TABLE IF NOT EXISTS CORE.MEMBER_PROFILE_USA
    LIKE CORE.MEMBER_PROFILE;

CREATE TABLE IF NOT EXISTS CORE.MEMBER_PROFILE_PHIL
    LIKE CORE.MEMBER_PROFILE;

CREATE TABLE IF NOT EXISTS CORE.MEMBER_PROFILE_CAN
    LIKE CORE.MEMBER_PROFILE;

CREATE TABLE IF NOT EXISTS CORE.MEMBER_PROFILE_AU
    LIKE CORE.MEMBER_PROFILE;

-- 3. Verify CORE tables
SHOW TABLES IN SCHEMA CORE;

DESCRIBE TABLE CORE.MEMBER_PROFILE;

In [ ]:
%%sql -r dataframe_3
-- =====================================================
-- PHASE 7A: RAW TO STG MEMBER TRANSFORMATION
-- NOTEBOOK: NB_03_MEMBER_PROCESSING
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

INSERT INTO STG.STG_MEMBER_PROFILES (
    MEMBER_NAME,
    MEMBER_ID,
    ENROLLMENT_DATE,
    LAST_FLIGHT_DATE,
    TIER_CODE,
    AGENT_NAME,
    STATE,
    COUNTRY,
    POST_CODE,
    DOB,
    ACTIVE_MEMBER,
    AGE,
    STALE_MEMBER,
    SOURCE_FILE_NAME,
    FILE_TIMESTAMP,
    INGESTION_TIMESTAMP,
    BATCH_ID,
    SOURCE_ROW_NUMBER,
    VALIDATION_STATUS,
    REJECTION_REASON
)
SELECT
    RAW_RECORD:MEMBER_NAME::VARCHAR(255),
    RAW_RECORD:MEMBER_ID::VARCHAR(18),

    TRY_TO_DATE(
        RAW_RECORD:ENROLLMENT_DATE::VARCHAR,
        'YYYYMMDD'
    ),

    TRY_TO_DATE(
        RAW_RECORD:LAST_FLIGHT_DATE::VARCHAR,
        'YYYYMMDD'
    ),

    RAW_RECORD:TIER_CODE::VARCHAR(5),
    RAW_RECORD:AGENT_NAME::VARCHAR(255),
    RAW_RECORD:STATE::VARCHAR(5),
    RAW_RECORD:COUNTRY::VARCHAR(5),
    TRY_TO_NUMBER(RAW_RECORD:POST_CODE::VARCHAR),

    TRY_TO_DATE(
        RAW_RECORD:DOB::VARCHAR,
        'DDMMYYYY'
    ),

    RAW_RECORD:ACTIVE_MEMBER::VARCHAR(1),

    DATEDIFF(
        YEAR,
        TRY_TO_DATE(RAW_RECORD:DOB::VARCHAR, 'DDMMYYYY'),
        CURRENT_DATE()
    )
    - IFF(
        TO_CHAR(CURRENT_DATE(), 'MMDD')
          < TO_CHAR(
                TRY_TO_DATE(RAW_RECORD:DOB::VARCHAR, 'DDMMYYYY'),
                'MMDD'
            ),
        1,
        0
    ),

    CASE
        WHEN TRY_TO_DATE(
            RAW_RECORD:LAST_FLIGHT_DATE::VARCHAR,
            'YYYYMMDD'
        ) IS NULL THEN NULL
        WHEN DATEDIFF(
            DAY,
            TRY_TO_DATE(
                RAW_RECORD:LAST_FLIGHT_DATE::VARCHAR,
                'YYYYMMDD'
            ),
            CURRENT_DATE()
        ) > 90 THEN TRUE
        ELSE FALSE
    END,

    SOURCE_FILE_NAME,
    FILE_TIMESTAMP,
    INGESTION_TIMESTAMP,
    BATCH_ID,
    SOURCE_ROW_NUMBER,

    CASE
        WHEN RAW_RECORD:MEMBER_NAME IS NULL THEN 'REJECTED'
        WHEN RAW_RECORD:MEMBER_ID IS NULL THEN 'REJECTED'
        WHEN TRY_TO_DATE(
            RAW_RECORD:ENROLLMENT_DATE::VARCHAR,
            'YYYYMMDD'
        ) IS NULL THEN 'REJECTED'
        ELSE 'VALID'
    END,

    CASE
        WHEN RAW_RECORD:MEMBER_NAME IS NULL
            THEN 'MEMBER_NAME_MISSING'
        WHEN RAW_RECORD:MEMBER_ID IS NULL
            THEN 'MEMBER_ID_MISSING'
        WHEN TRY_TO_DATE(
            RAW_RECORD:ENROLLMENT_DATE::VARCHAR,
            'YYYYMMDD'
        ) IS NULL THEN 'INVALID_ENROLLMENT_DATE'
        ELSE NULL
    END

FROM RAW.RAW_MEMBER_PROFILES
WHERE SOURCE_FILE_NAME = 'MEMBER_PROFILE_20261125_193100.csv';